# Interactive Client Value Storyboard — 11 Pages

## Main problem

**Which client segments generate the highest gross profit while maintaining strong customer satisfaction?**

The browser application presents the analysis as **11 separate story pages**. Only one page is visible at a time, preventing the charts from being compressed or crowded together. Use the page menu or the **Previous** and **Next** buttons to move through the story.

| Page | Story question |
|---:|---|
| 1 | What is the main problem and how will it be answered? |
| 2 | Which client type creates the greatest financial value? |
| 3 | What costs explain the differences in profitability? |
| 4 | Do profitable client types also report positive service experiences? |
| 5 | Where are the service strengths and gaps? |
| 6 | How does customer advocacy differ and change over time? |
| 7 | Which services should be improved or protected? |
| 8 | How much value is associated with Promoters, Passives and Detractors? |
| 9 | Where is advocacy risk concentrated? |
| 10 | Which clients should be retained, prioritised, improved or reconsidered? |
| 11 | Which segment best answers the main question? |

**Client Type is the default comparison throughout.** Open the collapsible filter panel to drill down by year, country, client type, service dimension or another client profile.


## Before running

Place **`merged_cleaned.xlsx`** in the same folder as this notebook. Run the cells from top to bottom. If a package is missing, install it once with:

```python
%pip install pandas numpy openpyxl plotly dash
```

The final cell contains the browser launch command. It is commented out so **Run All** can finish without starting a blocking server; remove the leading `#` only when you are ready to open the dashboard.


In [9]:
# 1. Imports, data preparation and validation
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, Input, Output, State, ctx, dcc, html


DATA_PATH = Path("merged_cleaned.xlsx")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Place merged_cleaned.xlsx in the same folder as this notebook, "
        "then run the cells again."
    )

raw = pd.read_excel(DATA_PATH)
raw.columns = raw.columns.astype(str).str.strip()

client_id_candidates = [
    "Client ID", "CLIENT ID", "ClientID", "CLIENTID",
    "client_id", "Client_Id", "Client Id"
]
client_id_column = next(
    (column for column in client_id_candidates if column in raw.columns),
    None
)
if client_id_column is None:
    raise KeyError(
        f"No client ID column was found. Checked: {client_id_candidates}"
    )

raw = raw.rename(columns={client_id_column: "Client ID"})

SERVICE_COLUMNS = [
    "PRESALES AND PARTNERSHIP",
    "TECHNICAL EXPERTISE",
    "PROJECT DELIVERY",
    "POST-SALES SUPPORT",
]

SERVICE_LABELS = {
    "PRESALES AND PARTNERSHIP": "Presales & Partnership",
    "TECHNICAL EXPERTISE": "Technical Expertise",
    "PROJECT DELIVERY": "Project Delivery",
    "POST-SALES SUPPORT": "Post-Sales Support",
}

SERVICE_COLOURS = {
    "Presales & Partnership": "#636EFA",
    "Technical Expertise": "#EF553B",
    "Project Delivery": "#00CC96",
    "Post-Sales Support": "#AB63FA",
}

NPS_ORDER = ["Promoter", "Passive", "Detractor"]
NPS_COLOURS = {
    "Promoter": "#168AAD",
    "Passive": "#F4A261",
    "Detractor": "#E76F51",
}

SEGMENT_LABELS = {
    "TYPE": "Client Type",
    "SECTOR": "Industry Sector",
    "STAFF STRENGTH": "Organisation Size",
    "COUNTRY": "Country",
}

CLIENT_PROFILE_MAP = {
    "TYPE": "Client_Type",
    "SECTOR": "Sector",
    "STAFF STRENGTH": "Organisation_Size",
    "COUNTRY": "Country",
}

required_columns = {
    "Client ID", "YEAR", "TYPE", "SECTOR", "STAFF STRENGTH",
    "COUNTRY", "REVENUE", "HARDWARE", "SOFTWARE", "MANPOWER",
    "NPS RATING", *SERVICE_COLUMNS,
}
missing_columns = sorted(required_columns - set(raw.columns))
if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

numeric_columns = [
    "YEAR", "REVENUE", "HARDWARE", "SOFTWARE", "MANPOWER",
    "NPS RATING", *SERVICE_COLUMNS,
]
for column in numeric_columns:
    raw[column] = pd.to_numeric(raw[column], errors="coerce")

analysis = raw.copy()
analysis["YEAR"] = analysis["YEAR"].astype("Int64")
analysis["Overall Satisfaction"] = analysis[SERVICE_COLUMNS].mean(axis=1)
analysis["COGS"] = analysis[[
    "HARDWARE", "SOFTWARE", "MANPOWER"
]].sum(axis=1, min_count=3)
analysis["Gross Profit"] = analysis["REVENUE"] - analysis["COGS"]
analysis["Gross Margin"] = (
    analysis["Gross Profit"]
    / analysis["REVENUE"].replace(0, np.nan)
)
analysis["NPS Category"] = np.select(
    [analysis["NPS RATING"].ge(9), analysis["NPS RATING"].ge(7)],
    ["Promoter", "Passive"],
    default="Detractor",
)
analysis["NPS Category"] = pd.Categorical(
    analysis["NPS Category"], categories=NPS_ORDER, ordered=True
)

years = sorted(analysis["YEAR"].dropna().astype(int).unique().tolist())
if not years:
    raise ValueError("YEAR contains no valid values.")

year_min, year_max = min(years), max(years)
client_types = sorted(analysis["TYPE"].dropna().astype(str).unique())
countries = sorted(analysis["COUNTRY"].dropna().astype(str).unique())

duplicate_client_years = int(
    analysis.duplicated(subset=["Client ID", "YEAR"]).sum()
)
if duplicate_client_years:
    raise ValueError(
        f"Found {duplicate_client_years} duplicate client-year records. "
        "Resolve them before using the storyboard."
    )

print(
    f"Loaded {len(analysis):,} client-year records, "
    f"{analysis['Client ID'].nunique():,} clients, "
    f"covering {year_min}–{year_max}."
)
print("Duplicate client-year records:", duplicate_client_years)


Loaded 392 client-year records, 140 clients, covering 2021–2025.
Duplicate client-year records: 0


In [10]:
# 2. Shared calculations, filters and dashboard components

GRAPH_CONFIG = {
    "displaylogo": False,
    "responsive": True,
    "scrollZoom": True,
    "toImageButtonOptions": {
        "format": "png",
        "filename": "client_value_storyboard",
        "scale": 2,
    },
}

PAGE_STYLE = {
    "fontFamily": "Segoe UI, Arial, sans-serif",
    "backgroundColor": "#F3F6FA",
    "color": "#172B4D",
    "minHeight": "100vh",
}

CONTENT_STYLE = {
    "maxWidth": "1500px",
    "margin": "0 auto",
    "padding": "0 22px 28px",
}

CARD_STYLE = {
    "backgroundColor": "white",
    "border": "1px solid #E3E8EF",
    "borderRadius": "12px",
    "boxShadow": "0 2px 8px rgba(23,43,77,0.06)",
}

def empty_figure(title, message, height=520):
    figure = go.Figure()
    figure.add_annotation(
        x=0.5,
        y=0.5,
        xref="paper",
        yref="paper",
        text=message,
        showarrow=False,
        font={"size": 16, "color": "#5E6C84"},
        align="center",
    )
    figure.update_layout(
        title={"text": title, "x": 0.5, "xanchor": "center"},
        template="plotly_white",
        height=height,
        xaxis={"visible": False},
        yaxis={"visible": False},
        margin={"t": 80, "r": 35, "b": 45, "l": 35},
    )
    return figure


def filter_rows(
    year_range,
    client_type="All",
    country="All",
    nps_categories=None,
):
    start_year, end_year = year_range
    filtered = analysis[
        analysis["YEAR"].between(start_year, end_year)
    ].copy()

    if client_type != "All":
        filtered = filtered[filtered["TYPE"].astype(str).eq(client_type)]
    if country != "All":
        filtered = filtered[
            filtered["COUNTRY"].astype(str).eq(country)
        ]

    if nps_categories is not None:
        if nps_categories:
            filtered = filtered[
                filtered["NPS Category"].astype(str).isin(nps_categories)
            ]
        else:
            filtered = filtered.iloc[0:0].copy()

    return filtered


def build_client_level(filtered_rows):
    """Aggregate filtered client-year rows to one record per client."""
    columns = [
        "Client ID", "Client_Type", "Sector", "Organisation_Size",
        "Country", "Revenue", "Gross_Profit", "Gross_Margin",
        "Average_Satisfaction", "Average_NPS", "First_Observed_Year",
        "Last_Observed_Year", "Observed_Longevity", "Client_Year_Records",
        "NPS Category", "Priority Group",
    ]
    if filtered_rows.empty:
        return pd.DataFrame(columns=columns)

    client = (
        filtered_rows
        .groupby("Client ID", as_index=False, observed=True)
        .agg(
            Client_Type=("TYPE", "first"),
            Sector=("SECTOR", "first"),
            Organisation_Size=("STAFF STRENGTH", "first"),
            Country=("COUNTRY", "first"),
            Revenue=("REVENUE", "sum"),
            Gross_Profit=("Gross Profit", "sum"),
            Average_Satisfaction=("Overall Satisfaction", "mean"),
            Average_NPS=("NPS RATING", "mean"),
            First_Observed_Year=("YEAR", "min"),
            Last_Observed_Year=("YEAR", "max"),
            Client_Year_Records=("YEAR", "size"),
        )
    )

    client["Gross_Margin"] = (
        client["Gross_Profit"]
        / client["Revenue"].replace(0, np.nan)
    )
    client["Observed_Longevity"] = (
        client["Last_Observed_Year"]
        - client["First_Observed_Year"]
        + 1
    )
    client["NPS Category"] = np.select(
        [client["Average_NPS"].ge(9), client["Average_NPS"].ge(7)],
        ["Promoter", "Passive"],
        default="Detractor",
    )
    client["NPS Category"] = pd.Categorical(
        client["NPS Category"], categories=NPS_ORDER, ordered=True
    )

    satisfaction_median = client["Average_Satisfaction"].median()
    margin_median = client["Gross_Margin"].median()
    client["Priority Group"] = np.select(
        [
            (
                client["Average_Satisfaction"].lt(satisfaction_median)
                & client["Gross_Margin"].ge(margin_median)
            ),
            (
                client["Average_Satisfaction"].ge(satisfaction_median)
                & client["Gross_Margin"].ge(margin_median)
            ),
            (
                client["Average_Satisfaction"].ge(satisfaction_median)
                & client["Gross_Margin"].lt(margin_median)
            ),
        ],
        ["Prioritise", "Retain", "Improve"],
        default="Reconsider",
    )
    return client[columns]


def profile_summary(filtered_rows, profile_column):
    if filtered_rows.empty:
        return pd.DataFrame()

    summary = (
        filtered_rows
        .groupby(profile_column, dropna=False, observed=True)
        .agg(
            Records=("Client ID", "size"),
            Clients=("Client ID", "nunique"),
            Revenue=("REVENUE", "sum"),
            Gross_Profit=("Gross Profit", "sum"),
            Average_Service=("Overall Satisfaction", "mean"),
            Average_NPS=("NPS RATING", "mean"),
        )
        .reset_index()
        .rename(columns={profile_column: "Profile"})
    )
    summary["Profile"] = summary["Profile"].fillna("Unknown").astype(str)
    summary["Gross_Margin"] = (
        summary["Gross_Profit"]
        / summary["Revenue"].replace(0, np.nan)
    )
    return summary


def filter_description(
    year_range,
    client_type,
    country,
    nps_categories,
):
    years_text = (
        str(year_range[0])
        if year_range[0] == year_range[1]
        else f"{year_range[0]}–{year_range[1]}"
    )
    type_text = "All client types" if client_type == "All" else client_type
    country_text = "All countries" if country == "All" else country
    nps_text = (
        "All advocacy groups"
        if set(nps_categories or []) == set(NPS_ORDER)
        else ", ".join(nps_categories or ["No advocacy groups"])
    )
    return f"{type_text} | {country_text} | {years_text} | {nps_text}"


def metric_card(label, value, context):
    return html.Div(
        [
            html.Div(
                label,
                style={
                    "fontSize": "12px",
                    "fontWeight": "700",
                    "color": "#5E6C84",
                    "textTransform": "uppercase",
                    "letterSpacing": "0.45px",
                },
            ),
            html.Div(
                value,
                style={
                    "fontSize": "24px",
                    "fontWeight": "750",
                    "color": "#172B4D",
                    "marginTop": "5px",
                },
            ),
            html.Div(
                context,
                style={
                    "fontSize": "12px",
                    "color": "#6B778C",
                    "marginTop": "5px",
                },
            ),
        ],
        style={**CARD_STYLE, "padding": "15px 17px"},
    )


def kpi_cards(filtered_rows):
    if filtered_rows.empty:
        values = [
            ("Gross profit", "—", "No matching records"),
            ("Weighted gross margin", "—", "No matching records"),
            ("Average satisfaction", "—", "Four service dimensions"),
            ("Average NPS rating", "—", "Individual 0–10 rating"),
        ]
    else:
        gross_profit = filtered_rows["Gross Profit"].sum()
        revenue = filtered_rows["REVENUE"].sum()
        gross_margin = gross_profit / revenue if revenue else np.nan
        values = [
            (
                "Gross profit",
                f"${gross_profit / 1_000_000:,.1f}M",
                f"Revenue ${revenue / 1_000_000:,.1f}M",
            ),
            (
                "Weighted gross margin",
                f"{gross_margin:.1%}" if pd.notna(gross_margin) else "—",
                "Total profit ÷ total revenue",
            ),
            (
                "Average satisfaction",
                f"{filtered_rows['Overall Satisfaction'].mean():.2f} / 5",
                "Mean of four service ratings",
            ),
            (
                "Average NPS rating",
                f"{filtered_rows['NPS RATING'].mean():.2f} / 10",
                (
                    f"{len(filtered_rows):,} records | "
                    f"{filtered_rows['Client ID'].nunique():,} clients"
                ),
            ),
        ]
    return [metric_card(*item) for item in values]


def graph_card(graph_id, evidence_label, question, guide, height=600):
    return html.Div(
        [
            html.Div(
                evidence_label,
                style={
                    "fontSize": "12px",
                    "fontWeight": "800",
                    "letterSpacing": "0.8px",
                    "color": "#0F6B78",
                    "textTransform": "uppercase",
                },
            ),
            html.H3(
                question,
                style={
                    "margin": "4px 0 5px",
                    "fontSize": "21px",
                    "color": "#172B4D",
                },
            ),
            html.P(
                guide,
                style={
                    "margin": "0 0 4px",
                    "fontSize": "13px",
                    "lineHeight": "1.45",
                    "color": "#5E6C84",
                },
            ),
            dcc.Graph(
                id=graph_id,
                config=GRAPH_CONFIG,
                style={"height": f"{height}px", "width": "100%"},
            ),
        ],
        style={**CARD_STYLE, "padding": "18px 20px 8px", "minWidth": 0},
    )


def transition_card(title, text):
    return html.Div(
        [
            html.Div(
                title,
                style={
                    "fontWeight": "800",
                    "color": "#0F6B78",
                    "marginBottom": "5px",
                },
            ),
            html.Div(
                text,
                style={"color": "#425466", "lineHeight": "1.55"},
            ),
        ],
        style={
            "backgroundColor": "#E9F5F5",
            "borderLeft": "5px solid #2A9D8F",
            "padding": "15px 18px",
            "borderRadius": "8px",
        },
    )


In [11]:
# 3. Profitability and cost-driver figures

def profit_figure(filtered_rows, profile_column):
    title = "Which client profiles create the most gross profit?"
    if filtered_rows.empty:
        return empty_figure(title, "No data match the selected filters.")

    summary = profile_summary(filtered_rows, profile_column)
    summary = summary.sort_values("Gross_Profit", ascending=True)
    chart_height = max(520, min(820, 42 * len(summary) + 190))

    figure = px.bar(
        summary,
        x="Gross_Profit",
        y="Profile",
        orientation="h",
        color=summary["Gross_Margin"] * 100,
        color_continuous_scale="Viridis",
        text="Gross_Profit",
        custom_data=[
            "Profile", "Revenue", "Gross_Margin", "Clients",
            "Records", "Average_Service", "Average_NPS",
        ],
        title=title,
    )
    figure.update_traces(
        texttemplate="$%{x:,.0f}",
        textposition="outside",
        cliponaxis=False,
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Gross profit: $%{x:,.0f}<br>"
            "Revenue: $%{customdata[1]:,.0f}<br>"
            "Weighted gross margin: %{customdata[2]:.1%}<br>"
            "Average satisfaction: %{customdata[5]:.2f}/5<br>"
            "Average NPS rating: %{customdata[6]:.2f}/10<br>"
            "Clients: %{customdata[3]}<br>"
            "Client-year records: %{customdata[4]}"
            "<extra></extra>"
        ),
    )
    figure.update_layout(
        template="plotly_white",
        height=chart_height,
        title={"x": 0.5, "xanchor": "center"},
        xaxis_title="Total gross profit ($)",
        yaxis_title=SEGMENT_LABELS[profile_column],
        coloraxis_colorbar={"title": "Weighted<br>gross margin (%)"},
        margin={"t": 80, "r": 100, "b": 65, "l": 150},
    )
    figure.update_xaxes(tickformat="$,.0f", rangemode="tozero")
    return figure


def cost_figure(filtered_rows, profile_column, display_mode):
    title = "Which costs are absorbing value within each profile?"
    if filtered_rows.empty:
        return empty_figure(title, "No data match the selected filters.")

    costs = (
        filtered_rows
        .groupby(profile_column, dropna=False, observed=True)[
            ["HARDWARE", "SOFTWARE", "MANPOWER"]
        ]
        .sum()
        .reset_index()
        .rename(columns={profile_column: "Profile"})
    )
    costs["Profile"] = costs["Profile"].fillna("Unknown").astype(str)
    costs["Total Cost"] = costs[
        ["HARDWARE", "SOFTWARE", "MANPOWER"]
    ].sum(axis=1)
    costs = costs.sort_values("Total Cost", ascending=True)
    chart_height = max(520, min(820, 42 * len(costs) + 190))

    cost_colours = {
        "HARDWARE": "#636EFA",
        "SOFTWARE": "#EF553B",
        "MANPOWER": "#00CC96",
    }
    figure = go.Figure()
    for column in ["HARDWARE", "SOFTWARE", "MANPOWER"]:
        figure.add_trace(
            go.Bar(
                x=costs[column],
                y=costs["Profile"],
                orientation="h",
                name=column.title(),
                marker_color=cost_colours[column],
                customdata=costs[["Total Cost"]].to_numpy(),
                hovertemplate=(
                    f"<b>%{{y}}</b><br>{column.title()}: $%{{x:,.0f}}<br>"
                    "Total cost: $%{customdata[0]:,.0f}<extra></extra>"
                ),
            )
        )

    figure.update_layout(
        title={"text": title, "x": 0.5, "xanchor": "center"},
        template="plotly_white",
        height=chart_height,
        barmode=display_mode,
        xaxis_title="Total cost ($)",
        yaxis_title=SEGMENT_LABELS[profile_column],
        legend={
            "title": "Cost component",
            "orientation": "h",
            "yanchor": "bottom",
            "y": 1.02,
            "xanchor": "center",
            "x": 0.5,
        },
        margin={"t": 110, "r": 40, "b": 65, "l": 150},
    )
    figure.update_xaxes(tickformat="$,.0f", rangemode="tozero")
    return figure


def value_experience_figure(filtered_rows, profile_column):
    title = "Which profiles combine healthy margins with strong service satisfaction?"
    if filtered_rows.empty:
        return empty_figure(title, "No data match the selected filters.")

    summary = profile_summary(filtered_rows, profile_column)
    if summary.empty:
        return empty_figure(title, "No profile summary can be calculated.")

    summary["Margin_Percent"] = summary["Gross_Margin"] * 100
    summary["Bubble_Value"] = summary["Revenue"].clip(lower=0)

    figure = px.scatter(
        summary,
        x="Average_Service",
        y="Margin_Percent",
        size="Bubble_Value",
        size_max=58,
        color="Gross_Profit",
        color_continuous_scale="Viridis",
        text="Profile",
        custom_data=[
            "Profile", "Gross_Profit", "Revenue", "Average_NPS",
            "Clients", "Records",
        ],
        title=title,
    )
    figure.update_traces(
        textposition="top center",
        marker={"line": {"width": 1, "color": "white"}, "opacity": 0.84},
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Average satisfaction: %{x:.2f}/5<br>"
            "Weighted gross margin: %{y:.1f}%<br>"
            "Gross profit: $%{customdata[1]:,.0f}<br>"
            "Revenue: $%{customdata[2]:,.0f}<br>"
            "Average NPS rating: %{customdata[3]:.2f}/10<br>"
            "Clients: %{customdata[4]}<br>"
            "Client-year records: %{customdata[5]}"
            "<extra></extra>"
        ),
    )

    satisfaction_median = summary["Average_Service"].median()
    margin_median = summary["Margin_Percent"].median()
    figure.add_vline(
        x=satisfaction_median,
        line_dash="dash",
        line_color="#5E6C84",
        annotation_text=f"Profile median {satisfaction_median:.2f}",
        annotation_position="top left",
    )
    figure.add_hline(
        y=margin_median,
        line_dash="dash",
        line_color="#5E6C84",
        annotation_text=f"Profile median {margin_median:.1f}%",
        annotation_position="bottom right",
    )
    figure.update_layout(
        template="plotly_white",
        height=610,
        title={"x": 0.5, "xanchor": "center"},
        xaxis_title="Average service satisfaction (1–5)",
        yaxis_title="Weighted gross margin (%)",
        coloraxis_colorbar={"title": "Gross profit ($)"},
        margin={"t": 85, "r": 90, "b": 70, "l": 80},
    )
    figure.update_xaxes(range=[1, 5.15])
    return figure


In [12]:
# 4. Satisfaction and loyalty figures

def service_figure(filtered_rows, selected_services, selected_period):
    title = "Where are service strengths and gaps?"
    if filtered_rows.empty:
        return empty_figure(title, "No data match the selected filters.")
    if not selected_services:
        return empty_figure(
            title, "Select at least one service dimension in the filter bar."
        )

    if selected_period == "Overall":
        chart_rows = filtered_rows.copy()
        period_label = "Filtered period overall"
    else:
        chart_rows = filtered_rows[
            filtered_rows["YEAR"].eq(int(selected_period))
        ].copy()
        period_label = str(selected_period)

    if chart_rows.empty:
        return empty_figure(
            title,
            "The selected service period is outside the current year filter.",
        )

    service_summary = (
        chart_rows
        .groupby("TYPE", observed=True)[selected_services]
        .mean()
        .reset_index()
    )
    counts = chart_rows.groupby("TYPE", observed=True).size()
    service_long = service_summary.melt(
        id_vars="TYPE",
        value_vars=selected_services,
        var_name="Service Column",
        value_name="Average Rating",
    )
    service_long["Service Dimension"] = service_long[
        "Service Column"
    ].map(SERVICE_LABELS)
    service_long["Records"] = service_long["TYPE"].map(counts)

    figure = px.bar(
        service_long,
        x="TYPE",
        y="Average Rating",
        color="Service Dimension",
        barmode="group",
        text="Average Rating",
        custom_data=["Records"],
        color_discrete_map=SERVICE_COLOURS,
        category_orders={
            "TYPE": client_types,
            "Service Dimension": [
                SERVICE_LABELS[column] for column in selected_services
            ],
        },
        title=f"{title} — {period_label}",
    )
    figure.update_traces(
        texttemplate="%{y:.2f}",
        textposition="outside",
        cliponaxis=False,
        hovertemplate=(
            "Client type: %{x}<br>"
            "Average rating: %{y:.2f}/5<br>"
            "Client-year records: %{customdata[0]}"
            "<extra>%{fullData.name}</extra>"
        ),
    )
    figure.update_layout(
        template="plotly_white",
        height=590,
        title={"x": 0.5, "xanchor": "center"},
        xaxis_title="Client type",
        yaxis={"title": "Average rating (1–5)", "range": [0, 5.35]},
        legend={
            "title": "Service dimension",
            "orientation": "h",
            "yanchor": "bottom",
            "y": 1.02,
            "xanchor": "center",
            "x": 0.5,
        },
        margin={"t": 120, "r": 35, "b": 65, "l": 70},
    )
    return figure


def nps_heatmap_figure(filtered_rows):
    title = "Which client types show the strongest customer advocacy?"
    if filtered_rows.empty:
        return empty_figure(title, "No data match the selected filters.")

    grouped = (
        filtered_rows
        .groupby(["TYPE", "YEAR"], observed=True)["NPS RATING"]
        .agg(["mean", "count"])
        .reset_index()
    )
    values = grouped.pivot(index="TYPE", columns="YEAR", values="mean")
    counts = grouped.pivot(index="TYPE", columns="YEAR", values="count")

    displayed_types = [
        item for item in client_types if item in values.index
    ]
    displayed_years = [item for item in years if item in values.columns]

    values = values.reindex(index=displayed_types, columns=displayed_years)
    counts = counts.reindex(index=displayed_types, columns=displayed_years)
    values["Overall"] = (
        filtered_rows.groupby("TYPE", observed=True)["NPS RATING"]
        .mean()
        .reindex(displayed_types)
    )
    counts["Overall"] = (
        filtered_rows.groupby("TYPE", observed=True).size()
        .reindex(displayed_types)
    )

    heatmap_text = np.array([
        ["" if pd.isna(value) else f"{value:.2f}" for value in row]
        for row in values.to_numpy()
    ])

    figure = go.Figure(
        go.Heatmap(
            z=values.to_numpy(),
            x=[str(column) for column in values.columns],
            y=values.index.tolist(),
            text=heatmap_text,
            texttemplate="%{text}",
            textfont={"size": 16},
            customdata=counts.to_numpy(),
            zmin=7,
            zmax=9,
            colorscale=[
                [0.00, "#E76F51"],
                [0.50, "#F4D35E"],
                [1.00, "#2A9D8F"],
            ],
            colorbar={"title": "Average<br>NPS rating"},
            hovertemplate=(
                "Client type: %{y}<br>"
                "Period: %{x}<br>"
                "Average NPS rating: %{z:.2f}/10<br>"
                "Client-year records: %{customdata:.0f}"
                "<extra></extra>"
            ),
        )
    )
    figure.update_layout(
        title={"text": title, "x": 0.5, "xanchor": "center"},
        template="plotly_white",
        height=560,
        xaxis_title="Period",
        yaxis_title="Client type",
        margin={"t": 85, "r": 90, "b": 70, "l": 85},
    )
    return figure


def service_priority_data(filtered_rows, selected_services):
    rows = []
    for column in selected_services:
        pair = filtered_rows[[column, "NPS RATING"]].dropna()
        enough_variation = (
            len(pair) >= 3
            and pair[column].nunique() >= 2
            and pair["NPS RATING"].nunique() >= 2
        )
        correlation = (
            pair[column].corr(pair["NPS RATING"], method="spearman")
            if enough_variation else np.nan
        )
        rows.append({
            "Service Column": column,
            "Service Dimension": SERVICE_LABELS[column],
            "Performance": float(pair[column].mean()) if len(pair) else np.nan,
            "Importance": float(correlation),
            "Records": len(pair),
        })

    priority = pd.DataFrame(rows).dropna(
        subset=["Performance", "Importance"]
    )
    if len(priority) < 2:
        return priority

    performance_cutoff = float(priority["Performance"].median())
    importance_cutoff = float(priority["Importance"].median())

    def assign_action(row):
        high_performance = row["Performance"] >= performance_cutoff
        high_importance = row["Importance"] >= importance_cutoff
        if not high_performance and high_importance:
            return "Improve First"
        if high_performance and high_importance:
            return "Protect Strength"
        if not high_performance and not high_importance:
            return "Monitor"
        return "Maintain"

    priority["Recommended Action"] = priority.apply(assign_action, axis=1)
    return priority


def service_priority_figure(filtered_rows, selected_services, segment_label):
    title = "What should be improved or protected first?"
    if filtered_rows.empty:
        return empty_figure(title, "No data match the selected filters.")
    if len(selected_services or []) < 2:
        return empty_figure(
            title,
            "Select at least two service dimensions to create relative priorities.",
        )

    priority = service_priority_data(filtered_rows, selected_services)
    if len(priority) < 2:
        return empty_figure(
            title,
            "The selected data do not contain enough variation for correlations.",
        )

    performance_cutoff = float(priority["Performance"].median())
    importance_cutoff = float(priority["Importance"].median())
    x_span = float(priority["Performance"].max() - priority["Performance"].min())
    y_span = float(priority["Importance"].max() - priority["Importance"].min())
    x_padding = max(0.16, x_span * 0.42)
    y_padding = max(0.08, y_span * 0.42)
    x_min = max(1.0, float(priority["Performance"].min() - x_padding))
    x_max = min(5.0, float(priority["Performance"].max() + x_padding))
    y_min = max(-1.0, float(priority["Importance"].min() - y_padding))
    y_max = min(1.0, float(priority["Importance"].max() + y_padding))
    if x_max - x_min < 0.35:
        midpoint = (x_min + x_max) / 2
        x_min, x_max = max(1, midpoint - 0.22), min(5, midpoint + 0.22)
    if y_max - y_min < 0.25:
        midpoint = (y_min + y_max) / 2
        y_min, y_max = max(-1, midpoint - 0.14), min(1, midpoint + 0.14)

    action_colours = {
        "Improve First": "#E76F51",
        "Protect Strength": "#2A9D8F",
        "Monitor": "#F4A261",
        "Maintain": "#457B9D",
    }
    text_positions = {
        "Presales & Partnership": "bottom left",
        "Technical Expertise": "top left",
        "Project Delivery": "top right",
        "Post-Sales Support": "bottom right",
    }

    figure = go.Figure()
    quadrants = [
        (x_min, performance_cutoff, importance_cutoff, y_max,
         "#FDE7E2", "Improve First", x_min, y_max, "left", "top"),
        (performance_cutoff, x_max, importance_cutoff, y_max,
         "#DDF2EC", "Protect Strength", x_max, y_max, "right", "top"),
        (x_min, performance_cutoff, y_min, importance_cutoff,
         "#FFF1DB", "Monitor", x_min, y_min, "left", "bottom"),
        (performance_cutoff, x_max, y_min, importance_cutoff,
         "#E6EEF7", "Maintain", x_max, y_min, "right", "bottom"),
    ]
    for x0, x1, y0, y1, colour, label, lx, ly, xa, ya in quadrants:
        figure.add_shape(
            type="rect", x0=x0, x1=x1, y0=y0, y1=y1,
            fillcolor=colour, opacity=0.62, line_width=0, layer="below",
        )
        figure.add_annotation(
            x=lx, y=ly, text=f"<b>{label}</b>", showarrow=False,
            xanchor=xa, yanchor=ya,
            xshift=10 if xa == "left" else -10,
            yshift=-9 if ya == "top" else 9,
            font={"size": 12, "color": "#42526E"},
        )

    for action in [
        "Improve First", "Protect Strength", "Monitor", "Maintain"
    ]:
        action_rows = priority[
            priority["Recommended Action"].eq(action)
        ]
        if action_rows.empty:
            continue
        figure.add_trace(
            go.Scatter(
                x=action_rows["Performance"],
                y=action_rows["Importance"],
                mode="markers+text",
                name=action,
                text=action_rows["Service Dimension"],
                textposition=[
                    text_positions.get(item, "top center")
                    for item in action_rows["Service Dimension"]
                ],
                marker={
                    "size": 18,
                    "color": action_colours[action],
                    "line": {"width": 2, "color": "white"},
                },
                customdata=action_rows[
                    ["Records", "Recommended Action"]
                ].to_numpy(dtype=object),
                hovertemplate=(
                    "<b>%{text}</b><br>"
                    "Average rating: %{x:.3f}/5<br>"
                    "Spearman correlation: %{y:+.3f}<br>"
                    "Valid records: %{customdata[0]}<br>"
                    "Recommended action: %{customdata[1]}"
                    "<extra></extra>"
                ),
            )
        )

    figure.add_vline(
        x=performance_cutoff, line_dash="dash",
        line_color="#5E6C84", line_width=2,
    )
    figure.add_hline(
        y=importance_cutoff, line_dash="dash",
        line_color="#5E6C84", line_width=2,
    )
    figure.update_layout(
        title={
            "text": (
                f"{title} — {segment_label}<br><sup>"
                "Performance = average rating; importance = association with NPS"
                "</sup>"
            ),
            "x": 0.5,
            "xanchor": "center",
        },
        template="plotly_white",
        height=620,
        xaxis={
            "title": "Current performance — average service rating (1–5)",
            "range": [x_min, x_max],
            "tickformat": ".2f",
            "nticks": 6,
        },
        yaxis={
            "title": "Relative importance — Spearman correlation with NPS",
            "range": [y_min, y_max],
            "tickformat": ".2f",
            "nticks": 6,
        },
        legend={
            "title": "Recommended action",
            "orientation": "h",
            "yanchor": "bottom",
            "y": 1.02,
            "xanchor": "center",
            "x": 0.5,
        },
        margin={"t": 125, "r": 55, "b": 80, "l": 90},
    )
    return figure


In [13]:
# 5. Advocacy and client-prioritisation figures

def advocacy_profit_figure(client_rows, profile_column):
    title = "Where is commercial value concentrated across advocacy groups?"
    if client_rows.empty:
        return empty_figure(title, "No clients match the selected filters.")

    base = client_rows.copy()
    base["Profile"] = base[profile_column].fillna("Unknown").astype(str)
    grouped = (
        base
        .groupby(["Profile", "NPS Category"], observed=True)
        .agg(
            Gross_Profit=("Gross_Profit", "sum"),
            Revenue=("Revenue", "sum"),
            Clients=("Client ID", "size"),
            Median_Longevity=("Observed_Longevity", "median"),
            Average_Satisfaction=("Average_Satisfaction", "mean"),
        )
        .reset_index()
    )
    grouped["Gross_Margin"] = (
        grouped["Gross_Profit"]
        / grouped["Revenue"].replace(0, np.nan)
    )
    grouped["NPS Category"] = pd.Categorical(
        grouped["NPS Category"], categories=NPS_ORDER, ordered=True
    )
    grouped = grouped.sort_values(
        ["NPS Category", "Gross_Profit"], ascending=[False, True]
    )
    grouped["Label"] = (
        grouped["Profile"].astype(str)
        + " — "
        + grouped["NPS Category"].astype(str)
    )
    grouped["Value Label"] = grouped["Gross_Profit"].map(
        lambda value: f"${value / 1_000_000:.1f}M"
    )
    chart_height = max(560, min(900, 35 * len(grouped) + 210))

    figure = px.bar(
        grouped,
        x="Gross_Profit",
        y="Label",
        orientation="h",
        color="NPS Category",
        text="Value Label",
        color_discrete_map=NPS_COLOURS,
        category_orders={"NPS Category": NPS_ORDER},
        custom_data=[
            "Profile", "NPS Category", "Revenue", "Gross_Profit",
            "Gross_Margin", "Clients", "Median_Longevity",
            "Average_Satisfaction",
        ],
        title=title,
    )
    figure.update_traces(
        textposition="outside",
        cliponaxis=False,
        hovertemplate=(
            "<b>%{customdata[0]} — %{customdata[1]}</b><br>"
            "Gross profit: $%{customdata[3]:,.0f}<br>"
            "Revenue: $%{customdata[2]:,.0f}<br>"
            "Weighted gross margin: %{customdata[4]:.1%}<br>"
            "Clients: %{customdata[5]}<br>"
            "Median observed longevity: %{customdata[6]:.1f} years<br>"
            "Average satisfaction: %{customdata[7]:.2f}/5"
            "<extra></extra>"
        ),
    )
    figure.update_layout(
        template="plotly_white",
        height=chart_height,
        title={"x": 0.5, "xanchor": "center"},
        xaxis_title="Total gross profit ($)",
        yaxis_title="",
        legend={
            "title": "Client-level advocacy category",
            "orientation": "h",
            "yanchor": "bottom",
            "y": 1.02,
            "xanchor": "center",
            "x": 0.5,
        },
        margin={"t": 115, "r": 55, "b": 70, "l": 210},
    )
    figure.update_xaxes(tickformat="$,.0f", rangemode="tozero")
    return figure


def advocacy_composition_figure(client_rows, profile_column):
    title = "Which profiles contain Promoters, Passives and Detractors?"
    if client_rows.empty:
        return empty_figure(title, "No clients match the selected filters.")

    base = client_rows.copy()
    base["Profile"] = base[profile_column].fillna("Unknown").astype(str)
    summary = (
        base.groupby("Profile", observed=True)
        .agg(
            Clients=("Client ID", "size"),
            Revenue=("Revenue", "sum"),
            Gross_Profit=("Gross_Profit", "sum"),
            Median_Longevity=("Observed_Longevity", "median"),
        )
    )
    counts = (
        base.groupby(["Profile", "NPS Category"], observed=True)
        .size()
        .unstack(fill_value=0)
    )
    counts.columns = counts.columns.astype(str)
    for category in NPS_ORDER:
        if category not in counts.columns:
            counts[category] = 0
    composition = summary.join(counts[NPS_ORDER], how="left")
    composition = composition.sort_values("Gross_Profit", ascending=True)

    figure = go.Figure()
    for category in NPS_ORDER:
        share = composition[category] / composition["Clients"] * 100
        customdata = np.column_stack([
            composition.index.to_numpy(),
            composition["Clients"].to_numpy(),
            composition["Gross_Profit"].to_numpy(),
            composition["Revenue"].to_numpy(),
            composition["Median_Longevity"].to_numpy(),
            composition[category].to_numpy(),
        ])
        figure.add_trace(
            go.Bar(
                x=share,
                y=composition.index,
                orientation="h",
                name=category,
                marker_color=NPS_COLOURS[category],
                text=[f"{value:.0f}%" if value >= 7 else "" for value in share],
                textposition="inside",
                customdata=customdata,
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    f"Advocacy category: {category}<br>"
                    "Share: %{x:.1f}%<br>"
                    "Clients in category: %{customdata[5]:.0f}<br>"
                    "Profile gross profit: $%{customdata[2]:,.0f}<br>"
                    "Total clients: %{customdata[1]:.0f}<br>"
                    "Median observed longevity: %{customdata[4]:.1f} years"
                    "<extra></extra>"
                ),
            )
        )

    annotations = []
    for profile, row in composition.iterrows():
        annotations.append(
            {
                "x": 101.5,
                "y": profile,
                "xref": "x",
                "yref": "y",
                "text": (
                    f"GP ${row['Gross_Profit'] / 1_000_000:.1f}M | "
                    f"n={int(row['Clients'])} | "
                    f"{row['Median_Longevity']:.1f}y"
                ),
                "showarrow": False,
                "xanchor": "left",
                "font": {"size": 11, "color": "#5E6C84"},
            }
        )

    chart_height = max(560, min(900, 42 * len(composition) + 220))
    figure.update_layout(
        title={"text": title, "x": 0.5, "xanchor": "center"},
        template="plotly_white",
        height=chart_height,
        barmode="stack",
        xaxis={
            "title": "Share of clients within profile (%)",
            "range": [0, 123],
            "ticksuffix": "%",
        },
        yaxis={
            "title": "",
            "categoryorder": "array",
            "categoryarray": composition.index.tolist(),
        },
        annotations=annotations,
        legend={
            "title": "Client-level advocacy category",
            "orientation": "h",
            "yanchor": "bottom",
            "y": 1.02,
            "xanchor": "center",
            "x": 0.5,
        },
        margin={"t": 115, "r": 210, "b": 70, "l": 150},
    )
    return figure


def client_priority_figure(client_rows):
    title = "Which clients should be retained, improved or reconsidered?"
    if client_rows.empty:
        return empty_figure(title, "No clients match the selected filters.")

    client_rows = client_rows.dropna(
        subset=["Average_Satisfaction", "Gross_Margin"]
    ).copy()
    if client_rows.empty:
        return empty_figure(title, "No valid satisfaction and margin pairs remain.")

    satisfaction_median = client_rows["Average_Satisfaction"].median()
    margin_median = client_rows["Gross_Margin"].median() * 100
    figure = go.Figure()

    for category in NPS_ORDER:
        category_rows = client_rows[
            client_rows["NPS Category"].astype(str).eq(category)
        ]
        if category_rows.empty:
            continue
        positive_profit = category_rows["Gross_Profit"].clip(lower=0)
        if positive_profit.max() > 0:
            marker_sizes = 10 + 30 * np.sqrt(
                positive_profit / positive_profit.max()
            )
        else:
            marker_sizes = np.full(len(category_rows), 12.0)

        customdata = category_rows[[
            "Client ID", "Client_Type", "Sector", "Country",
            "Revenue", "Gross_Profit", "Observed_Longevity",
            "Average_NPS", "Priority Group",
        ]].to_numpy(dtype=object)
        figure.add_trace(
            go.Scatter(
                x=category_rows["Average_Satisfaction"],
                y=category_rows["Gross_Margin"] * 100,
                mode="markers",
                name=category,
                marker={
                    "size": marker_sizes,
                    "color": NPS_COLOURS[category],
                    "opacity": 0.78,
                    "line": {"width": 0.9, "color": "white"},
                },
                customdata=customdata,
                hovertemplate=(
                    "<b>Client %{customdata[0]}</b><br>"
                    "Client type: %{customdata[1]}<br>"
                    "Sector: %{customdata[2]}<br>"
                    "Country: %{customdata[3]}<br>"
                    "Average satisfaction: %{x:.2f}/5<br>"
                    "Client-level gross margin: %{y:.1f}%<br>"
                    "Gross profit: $%{customdata[5]:,.0f}<br>"
                    "Average NPS: %{customdata[7]:.2f}/10<br>"
                    "Observed longevity: %{customdata[6]} years<br>"
                    "Action zone: %{customdata[8]}"
                    "<extra></extra>"
                ),
            )
        )

    figure.add_vline(
        x=satisfaction_median,
        line_dash="dash",
        line_color="#425466",
        line_width=2,
        annotation_text=f"Median satisfaction {satisfaction_median:.2f}",
        annotation_position="top right",
    )
    figure.add_hline(
        y=margin_median,
        line_dash="dash",
        line_color="#425466",
        line_width=2,
        annotation_text=f"Median margin {margin_median:.1f}%",
        annotation_position="bottom right",
    )

    x_min = max(1, client_rows["Average_Satisfaction"].min() - 0.15)
    x_max = min(5, client_rows["Average_Satisfaction"].max() + 0.15)
    y_values = client_rows["Gross_Margin"] * 100
    y_padding = max(4, (y_values.max() - y_values.min()) * 0.08)
    y_min, y_max = y_values.min() - y_padding, y_values.max() + y_padding
    quadrant_labels = [
        (x_min, y_max, "Prioritise", "left", "top", "#C23B31"),
        (x_max, y_max, "Retain", "right", "top", "#237A45"),
        (x_min, y_min, "Reconsider", "left", "bottom", "#7A4DD8"),
        (x_max, y_min, "Improve", "right", "bottom", "#2F6DB3"),
    ]
    for x, y, label, xa, ya, colour in quadrant_labels:
        figure.add_annotation(
            x=x, y=y, text=f"<b>{label}</b>", showarrow=False,
            xanchor=xa, yanchor=ya,
            xshift=10 if xa == "left" else -10,
            yshift=-10 if ya == "top" else 10,
            bgcolor="rgba(255,255,255,0.82)",
            bordercolor=colour,
            borderwidth=1,
            font={"size": 15, "color": colour},
        )

    figure.update_layout(
        title={"text": title, "x": 0.5, "xanchor": "center"},
        template="plotly_white",
        height=680,
        xaxis={
            "title": "Average overall satisfaction (1–5)",
            "range": [x_min, x_max],
        },
        yaxis={
            "title": "Client-level gross margin (%)",
            "range": [y_min, y_max],
        },
        legend={
            "title": "Client-level advocacy category",
            "orientation": "h",
            "yanchor": "bottom",
            "y": 1.02,
            "xanchor": "center",
            "x": 0.5,
        },
        margin={"t": 115, "r": 50, "b": 75, "l": 90},
    )
    return figure


In [14]:
# 6. Story text, dashboard layout and navigation

def evidence_summary(filtered_rows):
    if filtered_rows.empty:
        return html.Div("No records match the selected filters.")

    type_summary = profile_summary(filtered_rows, "TYPE")
    profit_leader = type_summary.loc[type_summary["Gross_Profit"].idxmax()]
    service_leader = type_summary.loc[type_summary["Average_Service"].idxmax()]
    nps_leader = type_summary.loc[type_summary["Average_NPS"].idxmax()]

    return html.Div(
        [
            html.Strong("Current evidence: "),
            html.Span(
                f"{profit_leader['Profile']} leads total gross profit "
                f"(${profit_leader['Gross_Profit'] / 1_000_000:.1f}M); "
                f"{service_leader['Profile']} has the strongest average "
                f"service satisfaction ({service_leader['Average_Service']:.2f}/5); "
                f"and {nps_leader['Profile']} has the highest average NPS "
                f"rating ({nps_leader['Average_NPS']:.2f}/10)."
            ),
        ],
        style={
            "backgroundColor": "#FFF9E8",
            "borderLeft": "5px solid #E6A700",
            "padding": "14px 18px",
            "borderRadius": "8px",
            "lineHeight": "1.55",
            "color": "#425466",
        },
    )


def main_conclusion(filtered_rows):
    if filtered_rows.empty:
        return html.Div("No conclusion can be calculated for an empty selection.")

    type_summary = profile_summary(filtered_rows, "TYPE")
    profit_leader = type_summary.loc[type_summary["Gross_Profit"].idxmax()]
    contribution_cutoff = type_summary["Gross_Profit"].median()
    financially_material = type_summary[
        type_summary["Gross_Profit"].ge(contribution_cutoff)
    ]
    balance_leader = financially_material.loc[
        financially_material["Average_Service"].idxmax()
    ]
    nps_leader = type_summary.loc[type_summary["Average_NPS"].idxmax()]

    leader_rows = filtered_rows[
        filtered_rows["TYPE"].astype(str).eq(str(profit_leader["Profile"]))
    ]
    service_diagnosis = service_priority_data(
        leader_rows, SERVICE_COLUMNS
    )
    performance_means = leader_rows[SERVICE_COLUMNS].mean()
    weakest_column = performance_means.idxmin()
    weakest_label = SERVICE_LABELS[weakest_column]
    if service_diagnosis.empty:
        associated_label = "the service dimensions with sufficient data"
    else:
        associated_label = service_diagnosis.loc[
            service_diagnosis["Importance"].idxmax(),
            "Service Dimension",
        ]

    if str(balance_leader["Profile"]) == str(profit_leader["Profile"]):
        balance_sentence = (
            f"{profit_leader['Profile']} also provides the strongest "
            "satisfaction balance among financially material client types."
        )
    else:
        balance_sentence = (
            f"{balance_leader['Profile']} provides the strongest balance "
            f"among client types at or above the median gross-profit "
            f"contribution, with satisfaction of "
            f"{balance_leader['Average_Service']:.2f}/5."
        )

    return html.Div(
        [
            html.H3(
                "Story 11 of 11 — Answer to the main question",
                style={"margin": "0 0 9px", "color": "#172B4D"},
            ),
            html.P(
                (
                    f"Within the current filters, {profit_leader['Profile']} "
                    f"generates the highest total gross profit at "
                    f"${profit_leader['Gross_Profit'] / 1_000_000:.1f}M, "
                    f"with average satisfaction of "
                    f"{profit_leader['Average_Service']:.2f}/5 and average "
                    f"NPS rating of {profit_leader['Average_NPS']:.2f}/10. "
                    f"{balance_sentence} {nps_leader['Profile']} records the "
                    f"highest advocacy level at {nps_leader['Average_NPS']:.2f}/10."
                ),
                style={"lineHeight": "1.6", "margin": "0 0 10px"},
            ),
            html.P(
                (
                    f"The commercial recommendation is to protect the "
                    f"high-value relationships while addressing "
                    f"{weakest_label}, the weakest service area for the "
                    f"gross-profit leader. At the same time, preserve "
                    f"{associated_label}, which has the strongest observed "
                    f"association with that segment's NPS rating. The final "
                    "client matrix should then be used to identify the "
                    "specific accounts to retain, improve, prioritise or reconsider."
                ),
                style={"lineHeight": "1.6", "margin": 0},
            ),
            html.Div(
                (
                    "Interpretation boundary: the dashboard identifies "
                    "descriptive patterns and associations. It does not prove "
                    "that a service change causes NPS, renewal or retention."
                ),
                style={
                    "fontSize": "12px",
                    "color": "#6B778C",
                    "marginTop": "11px",
                    "fontStyle": "italic",
                },
            ),
        ],
        style={
            **CARD_STYLE,
            "borderTop": "5px solid #2A9D8F",
            "padding": "20px 22px",
        },
    )


app = Dash(__name__, suppress_callback_exceptions=True)
app.title = "Client Value Storyboard — 11 Pages"

segment_options = [
    {"label": label, "value": column}
    for column, label in SEGMENT_LABELS.items()
]

STORY_PAGE_OPTIONS = [
    {"label": "01  Agenda", "value": 1},
    {"label": "02  Gross Profit", "value": 2},
    {"label": "03  Cost Drivers", "value": 3},
    {"label": "04  Margin + Service", "value": 4},
    {"label": "05  Service Ratings", "value": 5},
    {"label": "06  NPS Over Time", "value": 6},
    {"label": "07  Service Priority", "value": 7},
    {"label": "08  Advocacy Value", "value": 8},
    {"label": "09  Advocacy Mix", "value": 9},
    {"label": "10  Client Actions", "value": 10},
    {"label": "11  Final Answer", "value": 11},
]


def control_group(label, component):
    return html.Div(
        [
            html.Label(
                label,
                style={
                    "fontWeight": "700",
                    "fontSize": "12px",
                    "color": "#425466",
                    "display": "block",
                    "marginBottom": "5px",
                },
            ),
            component,
        ],
        style={"marginBottom": "13px"},
    )


def story_page(number, section, title, question, children):
    return html.Section(
        [
            html.Div(
                f"PAGE {number} OF 11  •  {section}",
                style={
                    "fontSize": "12px",
                    "fontWeight": "800",
                    "letterSpacing": "0.9px",
                    "color": "#0F6B78",
                    "textTransform": "uppercase",
                },
            ),
            html.H2(
                title,
                style={
                    "fontSize": "28px",
                    "lineHeight": "1.2",
                    "margin": "6px 0 5px",
                    "color": "#172B4D",
                },
            ),
            html.P(
                question,
                style={
                    "fontSize": "15px",
                    "lineHeight": "1.5",
                    "color": "#5E6C84",
                    "margin": "0 0 17px",
                },
            ),
            *children,
        ],
        id=f"story-page-{number}",
        style={"display": "block" if number == 1 else "none"},
    )


header = html.Div(
    [
        html.Div(
            "CLIENT VALUE STORYBOARD",
            style={
                "fontSize": "12px",
                "fontWeight": "800",
                "letterSpacing": "1.5px",
                "color": "#4DD0C8",
            },
        ),
        html.H1(
            "Which client segments create valuable and sustainable relationships?",
            style={
                "fontSize": "28px",
                "margin": "5px 0 7px",
                "color": "white",
            },
        ),
        html.Div(
            "Financial value → Satisfaction and loyalty → Advocacy → Client action",
            style={"color": "#D9E7F2", "fontSize": "15px"},
        ),
    ],
    style={
        "backgroundColor": "#172B4D",
        "padding": "22px 30px",
    },
)


filter_controls = html.Details(
    [
        html.Summary(
            "Filters and chart settings",
            style={
                "fontWeight": "800",
                "fontSize": "13px",
                "cursor": "pointer",
                "color": "#172B4D",
                "padding": "5px 0 10px",
            },
        ),
        control_group(
            "Client type",
            dcc.Dropdown(
                id="story-client-type",
                options=[{"label": "All client types", "value": "All"}]
                + [{"label": item, "value": item} for item in client_types],
                value="All",
                clearable=False,
            ),
        ),
        control_group(
            "Country",
            dcc.Dropdown(
                id="story-country",
                options=[{"label": "All countries", "value": "All"}]
                + [{"label": item, "value": item} for item in countries],
                value="All",
                clearable=False,
            ),
        ),
        control_group(
            "NPS categories",
            dcc.Dropdown(
                id="story-nps",
                options=[{"label": item, "value": item} for item in NPS_ORDER],
                value=NPS_ORDER,
                multi=True,
                clearable=False,
            ),
        ),
        control_group(
            "Service dimensions",
            dcc.Dropdown(
                id="story-services",
                options=[
                    {"label": SERVICE_LABELS[column], "value": column}
                    for column in SERVICE_COLUMNS
                ],
                value=SERVICE_COLUMNS,
                multi=True,
                clearable=False,
            ),
        ),
        control_group(
            "Observed years",
            dcc.RangeSlider(
                id="story-years",
                min=year_min,
                max=year_max,
                step=1,
                value=[year_min, year_max],
                marks={year: str(year) for year in years},
                allowCross=False,
            ),
        ),
        control_group(
            "Financial profile breakdown",
            dcc.Dropdown(
                id="profit-profile",
                options=segment_options,
                value="TYPE",
                clearable=False,
            ),
        ),
        control_group(
            "Cost display",
            dcc.RadioItems(
                id="cost-mode",
                options=[
                    {"label": "Stacked", "value": "stack"},
                    {"label": "Grouped", "value": "group"},
                ],
                value="stack",
                inline=True,
                labelStyle={"marginRight": "14px"},
            ),
        ),
        control_group(
            "Service-rating period",
            dcc.Dropdown(
                id="service-period",
                options=[{"label": "Filtered period overall", "value": "Overall"}]
                + [{"label": str(year), "value": str(year)} for year in years],
                value="Overall",
                clearable=False,
            ),
        ),
        control_group(
            "Advocacy profile breakdown",
            dcc.Dropdown(
                id="advocacy-profile",
                options=segment_options,
                value="TYPE",
                clearable=False,
            ),
        ),
        html.Button(
            "Reset filters",
            id="story-reset",
            n_clicks=0,
            style={
                "width": "100%",
                "height": "38px",
                "border": "none",
                "borderRadius": "7px",
                "backgroundColor": "#2A9D8F",
                "color": "white",
                "fontWeight": "750",
                "cursor": "pointer",
            },
        ),
    ],
    open=False,
    style={
        "borderTop": "1px solid #E3E8EF",
        "borderBottom": "1px solid #E3E8EF",
        "padding": "9px 0 13px",
        "margin": "14px 0",
    },
)


sidebar = html.Aside(
    [
        html.Div(
            "STORY PAGES",
            style={
                "fontSize": "12px",
                "fontWeight": "800",
                "letterSpacing": "1px",
                "color": "#5E6C84",
                "marginBottom": "8px",
            },
        ),
        dcc.RadioItems(
            id="story-page",
            options=STORY_PAGE_OPTIONS,
            value=1,
            inputStyle={"marginRight": "9px"},
            labelStyle={
                "display": "block",
                "padding": "7px 6px",
                "borderRadius": "6px",
                "fontSize": "13px",
                "cursor": "pointer",
            },
        ),
        filter_controls,
        html.Div(
            [
                html.Button(
                    "← Previous",
                    id="story-prev",
                    n_clicks=0,
                    disabled=True,
                    style={
                        "flex": 1,
                        "height": "38px",
                        "border": "1px solid #B9C3D0",
                        "borderRadius": "7px",
                        "backgroundColor": "white",
                        "fontWeight": "700",
                        "cursor": "pointer",
                    },
                ),
                html.Button(
                    "Next →",
                    id="story-next",
                    n_clicks=0,
                    style={
                        "flex": 1,
                        "height": "38px",
                        "border": "none",
                        "borderRadius": "7px",
                        "backgroundColor": "#172B4D",
                        "color": "white",
                        "fontWeight": "700",
                        "cursor": "pointer",
                    },
                ),
            ],
            style={"display": "flex", "gap": "8px"},
        ),
        html.Div(
            id="story-progress",
            children="Page 1 of 11",
            style={
                "fontSize": "12px",
                "textAlign": "center",
                "color": "#6B778C",
                "marginTop": "8px",
            },
        ),
    ],
    style={
        **CARD_STYLE,
        "padding": "18px",
        "position": "sticky",
        "top": "14px",
        "maxHeight": "calc(100vh - 28px)",
        "overflowY": "auto",
    },
)


agenda_page = story_page(
    1,
    "AGENDA",
    "How the evidence will answer the main problem",
    "Start with financial value, test whether the client experience is healthy, then connect advocacy to concrete client actions.",
    [
        html.Div(
            [
                html.Div(
                    [
                        html.Div("MAIN QUESTION", style={"fontWeight": "800", "fontSize": "12px", "color": "#0F6B78"}),
                        html.H3(
                            "Which client segments generate the highest gross profit while maintaining strong customer satisfaction?",
                            style={"fontSize": "23px", "lineHeight": "1.35", "margin": "8px 0 4px"},
                        ),
                    ],
                    style={**CARD_STYLE, "padding": "22px"},
                ),
                html.Div(
                    id="story-kpis",
                    style={
                        "display": "grid",
                        "gridTemplateColumns": "repeat(auto-fit, minmax(210px, 1fr))",
                        "gap": "14px",
                    },
                ),
                html.Div(id="story-evidence-summary"),
                html.Div(
                    [
                        metric_card("Phase 1", "Financial value", "Pages 2–4: profit, costs and margin"),
                        metric_card("Phase 2", "Satisfaction & loyalty", "Pages 5–7: service, NPS and priorities"),
                        metric_card("Phase 3", "Advocacy & action", "Pages 8–11: value, risk and decisions"),
                    ],
                    style={
                        "display": "grid",
                        "gridTemplateColumns": "repeat(auto-fit, minmax(220px, 1fr))",
                        "gap": "14px",
                    },
                ),
            ],
            style={"display": "grid", "gap": "16px"},
        )
    ],
)


profit_page = story_page(
    2,
    "FINANCIAL VALUE",
    "Locate the largest source of gross profit",
    "Which client profile creates the greatest financial contribution?",
    [
        graph_card(
            "profit-chart",
            "STORY 2 OF 11 — FINANCIAL CONTRIBUTION",
            "Which profiles create the most gross profit?",
            "Bar length shows total contribution. Use the weighted gross-margin colour and hover details to separate large revenue from healthy returns.",
            660,
        ),
        transition_card(
            "Next question",
            "A segment may generate high profit simply because it is large. Page 3 examines the cost structure behind that value.",
        ),
    ],
)


cost_page = story_page(
    3,
    "COST DRIVERS",
    "Explain what absorbs revenue before profit is created",
    "What costs help explain the differences in profitability?",
    [
        graph_card(
            "cost-chart",
            "STORY 3 OF 11 — COST PRESSURE",
            "Which costs are absorbing value within each profile?",
            "Compare hardware, software and manpower. Interpret the total cost together with the profile's gross profit and weighted margin.",
            660,
        ),
        transition_card(
            "Next question",
            "Financial value is not sustainable if the client experience is weak. Page 4 introduces satisfaction alongside margin.",
        ),
    ],
)


value_experience_page = story_page(
    4,
    "PROFIT–SATISFACTION BRIDGE",
    "Test whether financially attractive profiles also report good experiences",
    "Do profitable client profiles also maintain strong service satisfaction?",
    [
        graph_card(
            "value-experience-chart",
            "STORY 4 OF 11 — VALUE AND EXPERIENCE",
            "Which profiles combine healthy margins with strong service satisfaction?",
            "The upper-right combines stronger satisfaction and margin. Bubble size represents revenue; colour represents gross profit.",
            660,
        ),
        transition_card(
            "Next phase: understand the client experience",
            "Page 5 separates the overall experience into four service dimensions to reveal specific strengths and gaps.",
        ),
    ],
)


service_page = story_page(
    5,
    "SATISFACTION",
    "Separate the service experience into strengths and gaps",
    "Where are service ratings consistently strong or weak?",
    [
        graph_card(
            "service-chart",
            "STORY 5 OF 11 — SERVICE PERFORMANCE",
            "Where are service strengths and gaps?",
            "Compare selected dimensions on the same 1–5 scale. High ratings are strengths to protect; lower ratings identify potential gaps.",
            650,
        ),
        transition_card(
            "Next question",
            "Service satisfaction describes the experience. Page 6 checks whether clients are also willing to advocate for the company.",
        ),
    ],
)


nps_page = story_page(
    6,
    "LOYALTY AND ADVOCACY",
    "Track willingness to recommend across client types and years",
    "How does customer advocacy differ and change over time?",
    [
        graph_card(
            "nps-chart",
            "STORY 6 OF 11 — CUSTOMER ADVOCACY",
            "Which client types show the strongest average NPS ratings?",
            "The heatmap shows average individual NPS ratings, not the formal Net Promoter Score. Read printed values with the colours.",
            620,
        ),
        transition_card(
            "Next question",
            "Knowing that advocacy differs is not enough. Page 7 identifies which service areas should be improved or protected first.",
        ),
    ],
)


service_priority_page = story_page(
    7,
    "SERVICE ACTION",
    "Combine current performance with association to NPS",
    "Which service dimensions should be improved, protected, maintained or monitored?",
    [
        graph_card(
            "service-priority-chart",
            "STORY 7 OF 11 — SERVICE PRIORITY",
            "What should be improved or protected first?",
            "Performance is the average rating; importance is its Spearman association with NPS. Quadrants are relative priorities, not universal pass–fail standards.",
            670,
        ),
        transition_card(
            "Next phase: connect advocacy to money",
            "Page 8 tests how much commercial value is associated with Promoters, Passives and Detractors.",
        ),
    ],
)


advocacy_value_page = story_page(
    8,
    "ADVOCACY VALUE",
    "Measure the commercial contribution of each advocacy group",
    "How much gross profit is associated with Promoters, Passives and Detractors?",
    [
        graph_card(
            "advocacy-profit-chart",
            "STORY 8 OF 11 — ADVOCACY VALUE",
            "Where is commercial value concentrated across advocacy groups?",
            "High-value Passives are conversion opportunities, high-value Detractors may need recovery, and high-value Promoters should be protected.",
            680,
        ),
        transition_card(
            "Next question",
            "Page 9 changes the view from total dollars to the mix of Promoters, Passives and Detractors inside each profile.",
        ),
    ],
)


advocacy_mix_page = story_page(
    9,
    "ADVOCACY RISK",
    "Locate profiles with large Passive or Detractor shares",
    "Where is advocacy risk or unrealised relationship value concentrated?",
    [
        graph_card(
            "advocacy-composition-chart",
            "STORY 9 OF 11 — ADVOCACY MIX",
            "Which profiles contain Promoters, Passives and Detractors?",
            "Compare percentages with gross profit and client counts. A large Passive or Detractor share can signal unrealised value or relationship risk.",
            680,
        ),
        transition_card(
            "Next question",
            "Segment patterns guide strategy, but action occurs at account level. Page 10 identifies which individual clients merit attention.",
        ),
    ],
)


client_action_page = story_page(
    10,
    "CLIENT ACTION",
    "Move from segment patterns to account-level decisions",
    "Which clients should be retained, prioritised, improved or reconsidered?",
    [
        graph_card(
            "client-priority-chart",
            "STORY 10 OF 11 — CLIENT PRIORITY MATRIX",
            "Which clients require each management response?",
            "The matrix combines satisfaction, gross margin, gross profit and advocacy. Treat the median zones as screening guidance and inspect hover details before acting.",
            730,
        ),
        transition_card(
            "Final page",
            "Page 11 synthesises the evidence and directly answers the main problem under the active filters.",
        ),
    ],
)


conclusion_page = story_page(
    11,
    "FINAL ANSWER",
    "Translate the complete story into a segment recommendation",
    "Which client segment best combines financial value with a healthy customer relationship?",
    [html.Div(id="story-conclusion")],
)


page_container = html.Div(
    [
        agenda_page,
        profit_page,
        cost_page,
        value_experience_page,
        service_page,
        nps_page,
        service_priority_page,
        advocacy_value_page,
        advocacy_mix_page,
        client_action_page,
        conclusion_page,
    ]
)


app.layout = html.Div(
    [
        header,
        html.Div(
            [
                sidebar,
                html.Main(
                    [
                        html.Div(
                            id="story-filter-summary",
                            style={
                                "fontSize": "12px",
                                "color": "#5E6C84",
                                "margin": "0 2px 12px",
                            },
                        ),
                        page_container,
                    ],
                    style={"minWidth": 0},
                ),
            ],
            style={
                "display": "grid",
                "gridTemplateColumns": "260px minmax(0, 1fr)",
                "gap": "18px",
                "alignItems": "start",
                "maxWidth": "1580px",
                "margin": "0 auto",
                "padding": "18px 20px 28px",
            },
        ),
        html.Footer(
            "Descriptive decision support: average NPS rating is not the formal Net Promoter Score; association does not prove causation; observed longevity is not confirmed retention.",
            style={
                "padding": "15px 28px",
                "fontSize": "12px",
                "textAlign": "center",
                "color": "#6B778C",
                "backgroundColor": "white",
                "borderTop": "1px solid #E3E8EF",
            },
        ),
    ],
    style=PAGE_STYLE,
)


In [15]:
# 7. Dashboard callbacks

@app.callback(
    Output("story-kpis", "children"),
    Output("story-filter-summary", "children"),
    Output("story-evidence-summary", "children"),
    Output("profit-chart", "figure"),
    Output("cost-chart", "figure"),
    Output("value-experience-chart", "figure"),
    Output("service-chart", "figure"),
    Output("nps-chart", "figure"),
    Output("service-priority-chart", "figure"),
    Output("advocacy-profit-chart", "figure"),
    Output("advocacy-composition-chart", "figure"),
    Output("client-priority-chart", "figure"),
    Output("story-conclusion", "children"),
    Input("story-years", "value"),
    Input("story-client-type", "value"),
    Input("story-country", "value"),
    Input("story-nps", "value"),
    Input("story-services", "value"),
    Input("profit-profile", "value"),
    Input("cost-mode", "value"),
    Input("service-period", "value"),
    Input("advocacy-profile", "value"),
)
def update_storyboard(
    year_range,
    client_type,
    country,
    nps_categories,
    selected_services,
    profit_profile,
    cost_mode,
    service_period,
    advocacy_profile,
):
    nps_categories = nps_categories or []
    selected_services = selected_services or []

    filtered_rows = filter_rows(
        year_range,
        client_type,
        country,
        nps_categories,
    )

    # Client-level advocacy categories are assigned after aggregating the
    # selected years, so first retain all client-year NPS categories.
    client_source = filter_rows(
        year_range,
        client_type,
        country,
        NPS_ORDER,
    )
    client_rows = build_client_level(client_source)
    if nps_categories:
        client_rows = client_rows[
            client_rows["NPS Category"].astype(str).isin(nps_categories)
        ].copy()
    else:
        client_rows = client_rows.iloc[0:0].copy()

    segment_label = (
        "Overall"
        if client_type == "All"
        else client_type
    )
    client_profile = CLIENT_PROFILE_MAP[advocacy_profile]

    return (
        kpi_cards(filtered_rows),
        "Showing: " + filter_description(
            year_range, client_type, country, nps_categories
        ),
        evidence_summary(filtered_rows),
        profit_figure(filtered_rows, profit_profile),
        cost_figure(filtered_rows, profit_profile, cost_mode),
        value_experience_figure(filtered_rows, profit_profile),
        service_figure(
            filtered_rows, selected_services, service_period
        ),
        nps_heatmap_figure(filtered_rows),
        service_priority_figure(
            filtered_rows, selected_services, segment_label
        ),
        advocacy_profit_figure(client_rows, client_profile),
        advocacy_composition_figure(client_rows, client_profile),
        client_priority_figure(client_rows),
        main_conclusion(filtered_rows),
    )


@app.callback(
    Output("story-years", "value"),
    Output("story-client-type", "value"),
    Output("story-country", "value"),
    Output("story-nps", "value"),
    Output("story-services", "value"),
    Output("profit-profile", "value"),
    Output("cost-mode", "value"),
    Output("service-period", "value"),
    Output("advocacy-profile", "value"),
    Input("story-reset", "n_clicks"),
    prevent_initial_call=True,
)
def reset_storyboard(_clicks):
    return (
        [year_min, year_max],
        "All",
        "All",
        NPS_ORDER,
        SERVICE_COLUMNS,
        "TYPE",
        "stack",
        "Overall",
        "TYPE",
    )




@app.callback(
    Output("story-page", "value"),
    Input("story-prev", "n_clicks"),
    Input("story-next", "n_clicks"),
    State("story-page", "value"),
    prevent_initial_call=True,
)
def navigate_story(_previous_clicks, _next_clicks, current_page):
    current_page = int(current_page or 1)
    if ctx.triggered_id == "story-prev":
        return max(1, current_page - 1)
    if ctx.triggered_id == "story-next":
        return min(11, current_page + 1)
    return current_page


@app.callback(
    Output("story-page-1", "style"),
    Output("story-page-2", "style"),
    Output("story-page-3", "style"),
    Output("story-page-4", "style"),
    Output("story-page-5", "style"),
    Output("story-page-6", "style"),
    Output("story-page-7", "style"),
    Output("story-page-8", "style"),
    Output("story-page-9", "style"),
    Output("story-page-10", "style"),
    Output("story-page-11", "style"),
    Output("story-prev", "disabled"),
    Output("story-next", "disabled"),
    Output("story-progress", "children"),
    Input("story-page", "value"),
)
def display_story_page(selected_page):
    selected_page = int(selected_page or 1)
    page_styles = [
        {"display": "block"} if page == selected_page else {"display": "none"}
        for page in range(1, 12)
    ]
    return (
        *page_styles,
        selected_page == 1,
        selected_page == 11,
        f"Page {selected_page} of 11",
    )


# Build the default figures once as a smoke check without starting a server.
_default_rows = filter_rows(
    [year_min, year_max], "All", "All", NPS_ORDER
)
_default_clients = build_client_level(_default_rows)
_smoke_figures = [
    profit_figure(_default_rows, "TYPE"),
    cost_figure(_default_rows, "TYPE", "stack"),
    value_experience_figure(_default_rows, "TYPE"),
    service_figure(_default_rows, SERVICE_COLUMNS, "Overall"),
    nps_heatmap_figure(_default_rows),
    service_priority_figure(_default_rows, SERVICE_COLUMNS, "Overall"),
    advocacy_profit_figure(_default_clients, "Client_Type"),
    advocacy_composition_figure(_default_clients, "Client_Type"),
    client_priority_figure(_default_clients),
]
assert all(isinstance(figure, go.Figure) for figure in _smoke_figures)
print("Storyboard ready: 11 separate pages, 9 figures, live filters, and a dynamic final conclusion.")


Storyboard ready: 11 separate pages, 9 figures, live filters, and a dynamic final conclusion.


## Open the storyboard in a browser

1. Run every cell above in order.
2. Run the launch cell below.
3. VS Code or Jupyter will print a local address such as `http://127.0.0.1:8050/`. Click it or paste it into your browser.

If port 8050 is already in use, change it to 8051. Stop the notebook cell or kernel when the presentation is finished.


In [16]:
# Presentation launch command — uncomment and run this cell manually.
app.run(debug=False, jupyter_mode="external", port=8050)


Dash app running on http://127.0.0.1:8050/
